In [1]:
import os 

In [2]:
%pwd

'd:\\PredictBot-Score-MLOps\\research'

In [3]:

# ".." tells Python to step out of the current folder into the parent folder
os.chdir("..")

# Check where you are now


In [4]:
%pwd


'd:\\PredictBot-Score-MLOps'

In [5]:
from src.predictor_bot_score.constants import CONFIG_PATH
from src.predictor_bot_score.logger import logger
from src.predictor_bot_score.utils.common import yaml_load , create_directories
from dataclasses import dataclass
from pathlib import Path
from datetime import datetime
import os
import boto3
import io
import json 
import pandas as pd
from pathlib import Path
from botocore.exceptions import (
    ClientError,
    NoCredentialsError,
    PartialCredentialsError,
    EndpointConnectionError,
    ConnectTimeoutError,
)



In [6]:
@dataclass(frozen=True)
class Data_injestion_config:

    bucket_name : str 
    file_name  : str
    raw_data : Path
    ingested_data : Path



In [7]:
class yaml_configruation:

    def __init__(self,config_path = CONFIG_PATH):
        self.config_path = yaml_load(config_path)

        create_directories([self.config_path.artifacts_root])

    def get_data_ingestion_config(self)-> Data_injestion_config:

        config_path = self.config_path.data_ingestion

        create_directories([config_path.raw_data,config_path.ingested_data])

        return Data_injestion_config(
            bucket_name = config_path.bucket_name,
            file_name = config_path.file_name,
            raw_data = config_path.raw_data,
            ingested_data = config_path.ingested_data

        )


In [ ]:
class Data_injestion:

    def __init__(self, config : Data_injestion_config):

        self.config = config
        self.s3 =   boto3.client(
                        's3', 
                        aws_access_key_id= os.getenv("AWS_ACCESS_KEY_ID"),
                        aws_secret_access_key=os.getenv("AWS_SECRET_ACCESS_KEY"))
        self.BUCKET_NAME = self.config.bucket_name
        self.FILE_NAME = self.config.file_name
        self.pipeline_run_id = datetime.now().strftime("%Y_%m_%d_%H_%M")

    def connection_check(self):

        try:

            response = self.s3.list_objects_v2(Bucket=self.BUCKET_NAME)
            return True

        except ClientError as c:
            # Extract the specific AWS error code dictionary string
            error_code = c.response['Error']['Message']
            print( error_code)
            raise
            
    
    def lambda_file_check(self):

        try:
            file_keys = []
            if self.connection_check() :
                logger.info("S3 . Connection exists ")
                response  = self.s3.list_objects_v2(Bucket = self.BUCKET_NAME,Prefix='lambda-api-data')

                if 'Contents' in response:
                    for obj in response.get('Contents',[]):
                        if obj['Key'].endswith(".csv"):
                            file_keys.append(obj['Key'])
                            logger.info(f"Found {len(file_keys)} files in 'lambda-api-data/'.")
                        else:
                            logger.warning("No files found under the prefix 'lambda-api-data/'.")
            return file_keys

        except ClientError as e:
            logger.error(f"S3 connection error: {e.response['Error']['Message']}")
            raise

    def raw_data_file_check(self):

        try:
            if self.connection_check() :
                logger.info("S3 . Connection exists ")
                response  = self.s3.list_objects_v2(Bucket = self.BUCKET_NAME,Prefix='raw-master-data')

                if 'Contents' in response:
                    all_keys = [i['Key'] for i in response['Contents']]

                    if self.FILE_NAME in all_keys:
                        logger.info(f"File {self.FILE_NAME} found in raw-master-data.")
                        
                        raw_data_obj = self.s3.get_object(
                            Bucket=self.BUCKET_NAME, 
                            Key=self.FILE_NAME)
                        raw_df = pd.read_csv((raw_data_obj['Body']))
                    
            return raw_df
        except ClientError as e:
            logger.error(f"S3 connection error: {e.response['Error']['Message']}")
            raise

    def lambda_conmbine_files(self,key):
        try:
            obj = self.s3.get_object(Bucket='predict-bot-mlops',Key=key)
            df = pd.read_csv(io.BytesIO(obj['Body'].read()))
            df = df.rename(columns={'value':'bot_score'})

            print(f"Read {len(df)} rows from {key}")

            return df
        except ClientError as e:
            print(f"Failed to read {key}: {e.response['Error']['Message']}")
            raise
    
    def combine_files(self ,keys):
        try:
            dfs = []
            for key in keys:
                df = self.lambda_conmbine_files(key)
                if df is not None:
                    dfs.append(df)

            if not dfs:
                logger.warning("No dataframes to combine.")
                return None
            
            combined = pd.concat(dfs, ignore_index=True)
            logger.info(f"Successfully combined {len(dfs)} files.")

            logger.info(f"Successfully combined {len(dfs)} files. ")
            return combined

        except Exception as e:
            logger.error(f"Error during file combination: {e}")
            raise

 
    def save_file_s3(self ,df , output_key):
        try:
            buf = io.StringIO()
            df.to_csv(buf, index=False)
            self.s3.put_object(
                Bucket = self.BUCKET_NAME,
                Key =   output_key,
                Body         = buf.getvalue(),
                ContentType  = 'text/csv'
            )
            logger.info(f"Saved {len(df)} rows → s3://{self.BUCKET_NAME}/{output_key}")
        except ClientError as e:
            logger.error(f"S3 save failed: {e.response['Error']['Message']}")
            raise

    def save_manifest(self,output_key):
        try:
            
            manifest = {
                "pipeline_run_id" : self.pipeline_run_id,
                "output_key"      : output_key,
                "completed_at"    : datetime.now().isoformat(),
                "status"          : "success",
            }
                
            manifest_key = f"combined_data/run__{self.pipeline_run_id}/manifest.json"
            local_dir = os.path.join(self.config.ingested_data,
                                     f"lambda_run__{self.pipeline_run_id}")
            
            local_path = os.path.join(local_dir,"manifest.json")
            with open(local_path , 'w') as f:
                json.dump(manifest ,f ,indent=4)
                logger.info(f"Manifest saved locally  {local_dir}")

        except ClientError as e:
            logger.error(f"Manifest S3 save failed: {e.response['Error']['Message']}")
            raise
        except Exception as e:
            logger.error(f"Manifest local save failed: {e}")
            raise
    
    def save_local(self ,df):
        try:
            local_dir = os.path.join(self.config.ingested_data,
                                     f"lambda_run__{self.pipeline_run_id}"
        )
            os.makedirs(local_dir ,exist_ok=True)
            local_path = os.path.join(local_dir, "combined.csv")
            df.to_csv(local_path, index=False)

            logger.info(f"Saved locally ****  {local_path}")
            return local_path

        except Exception as e:
            logger.error(f"Local save failed: {e}")
            raise

           

    def run(self):

        try:
            logger.info("=" * 50)
            logger.info("DATA INGESTION PIPELINE STARTED")
            logger.info(f"Run ID : {self.pipeline_run_id}")
            logger.info("=" * 50)

            self.connection_check()

            raw_df  = self.raw_data_file_check()

            lambda_keys  = self.lambda_file_check()
            lamdat_combined_df = self.combine_files(lambda_keys)

            if raw_df is None and lamdat_combined_df is None:
                    raise FileNotFoundError(
                        "No data found — upload master data to raw-master-data/ "
                        "and ensure Lambda has run at least once"
                    )
            
            combined_df = pd.concat([raw_df , lamdat_combined_df],ignore_index=True)

            output_key = f"combined_data/run__{self.pipeline_run_id}/combined_df.csv"
            self.save_local(combined_df)

            self.save_manifest(output_key)

            logger.info("=" * 50)
            logger.info(f"DATA INGESTION COMPLETE")
            logger.info(f"Total rows : {len(combined_df)}")
            logger.info(f"Output     : s3://{self.BUCKET_NAME}/{output_key}")
            logger.info("=" * 50)

            

        except FileNotFoundError as e:
            logger.error(f"Ingestion failed — missing data: {e}")
            raise
        except ClientError as e:
            logger.error(f"Ingestion failed — S3 error: {e.response['Error']['Message']}")
            raise
        except Exception as e:
            logger.error(f"Ingestion failed: {e}")
            raise


    



            




In [11]:
con = yaml_configruation()
con_data_injestion = con.get_data_ingestion_config()
con_data_injestion = Data_injestion(con_data_injestion)
con_data_injestion.run()


[2026-07-06 20:33:21,764: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-07-06 20:33:21,771: INFO: common: Directory created (or already exists) at: artifacts]
[2026-07-06 20:33:21,772: INFO: common: Directory created (or already exists) at: artifacts/data_ingestion/raw_data]
[2026-07-06 20:33:21,775: INFO: common: Directory created (or already exists) at: artifacts/data_ingestion/ingested_data]
[2026-07-06 20:33:21,792: INFO: 1142629072: ==================================================]
[2026-07-06 20:33:21,793: INFO: 1142629072: DATA INGESTION PIPELINE STARTED]
[2026-07-06 20:33:21,794: INFO: 1142629072: Run ID : 2026_07_06_20_33]
[2026-07-06 20:33:21,794: INFO: 1142629072: ==================================================]
[2026-07-06 20:33:24,252: INFO: 1142629072: S3 . Connection exists ]
[2026-07-06 20:33:24,575: INFO: 1142629072: File raw-master-data/Cloud_fare_Master_data.csv found in raw-master-data.]
[2026-07-06 20:33:33,623: INFO: 1142629072: S3 . 

,timestamp,bot_score
0,2025-01-01 00:00:00+00:00,1.000000
1,2025-01-01 00:15:00+00:00,0.802741
2,2025-01-01 00:30:00+00:00,0.806656
3,2025-01-01 00:45:00+00:00,0.782846
4,2025-01-01 01:00:00+00:00,0.787702
...,...,...
40753,2026-07-03 23:00:00+00:00,0.956689
40754,2026-07-04 00:00:00+00:00,0.970677
40755,2026-07-04 01:00:00+00:00,0.947923
40756,2026-07-04 02:00:00+00:00,0.929918
